In [ ]:
predictions_df = unseen_df[['MatibabuID', 'PredictedRiskCategory']].copy()

deduplicated_df = pd.merge(deduplicated_df, predictions_df, on='MatibabuID', how='left')

score_3_df = deduplicated_df[deduplicated_df['SelectedTotalScore'] == 3].copy()

misclassified_score_3 = score_3_df[(score_3_df['RiskCategory'] == 'low risk') & (score_3_df['PredictedRiskCategory'] == 'high risk')].copy()

print("Instances with SelectedTotalScore = 3 that were misclassified as 'high risk':")
display(misclassified_score_3[['ComplicationRiskScore', 'conditionScore', 'PPIScorevalue', 'AgeScore', 'PregnantBeforeScore', 'RiskCategory', 'PredictedRiskCategory']])

print("\nAnalysis of features for misclassified instances (SelectedTotalScore = 3):")
display(misclassified_score_3[['ComplicationRiskScore', 'conditionScore', 'PPIScorevalue', 'AgeScore', 'PregnantBeforeScore']].describe())

In [ ]:
# Filter where SelectedTotalScore = 3
score_3_low_risk_expected = deduplicated_df[deduplicated_df['SelectedTotalScore'] == 3].copy()

# Decide which prediction column to use
if 'PredictedRiskCategory_NewModel' in score_3_low_risk_expected.columns:
    pred_col = 'PredictedRiskCategory_NewModel'
elif 'PredictedRiskCategory' in score_3_low_risk_expected.columns:
    pred_col = 'PredictedRiskCategory'
else:
    raise KeyError("No predicted risk category column found in DataFrame!")

print("Instances with SelectedTotalScore = 3 and their predicted risk categories:")
display(score_3_low_risk_expected[['SelectedTotalScore', 'RiskCategory', pred_col]].head())

print("\nValue counts of predicted risk categories for instances with SelectedTotalScore = 3:")
display(score_3_low_risk_expected[pred_col].value_counts())

# Compare expected vs actual
expected_low_risk_count = len(score_3_low_risk_expected)
actual_low_risk_predicted_count = score_3_low_risk_expected[pred_col].value_counts().get('low risk', 0)

print(f"\nExpected number of instances with SelectedTotalScore = 3 to be 'low risk': {expected_low_risk_count}")
print(f"Actual number of instances with SelectedTotalScore = 3 predicted as 'low risk': {actual_low_risk_predicted_count}")

if expected_low_risk_count == actual_low_risk_predicted_count:
    print("\nThe model's predictions for score 3 instances now align with the score-based logic.")
else:
    print("\nThe model's predictions for score 3 instances do NOT fully align with the score-based logic.")
    misclassified_score_3_after_resampling = score_3_low_risk_expected[
        score_3_low_risk_expected[pred_col] != 'low risk'
    ].copy()

    print("\nInstances with SelectedTotalScore = 3 misclassified after resampling:")
    display(misclassified_score_3_after_resampling[['SelectedTotalScore', 'RiskCategory', pred_col]])


In [ ]:
import pandas as pd
import numpy as np

file_path_test_data = '/content/drive/MyDrive/mek.csv'

try:
    test_df = pd.read_csv(file_path_test_data)
    print(f"Successfully loaded test data from {file_path_test_data}")
    required_columns = ['ComplicationRiskScore', 'conditionScore', 'PPIScorevalue', 'AgeScore', 'PregnantBeforeScore']
    X_test_unseen = test_df[required_columns]

    X_test_unseen_preprocessed = preprocessor.transform(X_test_unseen)

    prediction_probs_test_unseen = model.predict(X_test_unseen_preprocessed)

    predicted_class_index_test_unseen = np.argmax(prediction_probs_test_unseen, axis=1)

    predicted_risk_category_test_unseen = label_encoder.inverse_transform(predicted_class_index_test_unseen)

    test_df['PredictedRiskCategory'] = predicted_risk_category_test_unseen

    print("\nTest data with predicted risk categories:")
    display(test_df.head())

    print("\nValue counts of predicted risk categories on test data:")
    display(test_df['PredictedRiskCategory'].value_counts())

except FileNotFoundError:
    print(f"Error: The file was not found at {file_path_test_data}")
except NameError:
    print("Error: 'preprocessor', 'model', or 'label_encoder' are not defined. Please ensure you have run the cells that define and train the model and fit the preprocessor and label encoder.")
except KeyError as e:
    print(f"Error: Missing required column in the CSV file: {e}. Please ensure the file contains the columns used for training: {required_columns}")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd
import numpy as np

file_path_test_data = '/content/drive/MyDrive/reduced_pregnancy_data.csv'

try:
    test_df = pd.read_csv(file_path_test_data)
    print(f"Successfully loaded test data from {file_path_test_data}")

    required_columns = ['ComplicationRiskScore', 'conditionScore', 'PPIScorevalue', 'AgeScore', 'PregnantBeforeScore']
    X_test_unseen = test_df[required_columns]

    X_test_unseen_preprocessed = preprocessor.transform(X_test_unseen)

    prediction_probs_test_unseen = model.predict(X_test_unseen_preprocessed)
    predicted_class_index_test_unseen = np.argmax(prediction_probs_test_unseen, axis=1)
    predicted_risk_category_test_unseen = label_encoder.inverse_transform(predicted_class_index_test_unseen)

    test_df['PredictedRiskCategory'] = predicted_risk_category_test_unseen

    print("\nTest data with predicted risk categories:")
    display(test_df.head())

    print("\nValue counts of predicted risk categories on test data:")
    display(test_df['PredictedRiskCategory'].value_counts())

except FileNotFoundError:
    print(f"Error: The file was not found at {file_path_test_data}")
except NameError:
    print("Error: 'preprocessor', 'model', or 'label_encoder' are not defined. Please ensure you have run the cells that define and train the model and fit the preprocessor and label encoder.")
except KeyError as e:
    print(f"Error: Missing required column in the CSV file: {e}. Please ensure the file contains the columns used for training: {required_columns}")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ✅ Step 1: Prepare your data
# Assuming X_resampled and y_resampled are already defined
# Split test set
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# ✅ Step 2: Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# ✅ Step 3: Compute class weights
num_classes = len(np.unique(y_train_encoded))
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)
class_weight_dict = dict(enumerate(class_weights))

# ✅ Step 4: Build the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()

# ✅ Step 5: Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ✅ Step 6: Define callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-5
)

# ✅ Step 7: Train the model
history = model.fit(
    X_train, y_train_encoded,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1
)

# ✅ Step 8: Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test_encoded)
print("Model evaluation results on original test set:")
print("Loss:", loss)
print("Accuracy:", accuracy)
